# Bank Telemarketing Data Workflow: Cleaning and Exploratory Analysis

**Author:** Lampros Gidarakos
**Dataset:** UCI Bank Marketing, file `bank-additional-full.csv` (Moro, Rita & Cortez), 41,188 phone contacts from a Portuguese bank's term-deposit campaigns, May 2008 to November 2010.

This notebook builds a reproducible, function-based workflow that loads the raw campaign data, audits its quality, cleans it with documented decisions, and explores which client and campaign characteristics are associated with a client subscribing to a term deposit (`y`). The result is an analysis-ready table and three figures that a marketing or analytics team could use before any modelling.

In [1]:
# Imports and settings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
sns.set_theme(style="whitegrid")

DATA_PATH = Path("data/bank-additional-full.csv")
FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
print("pandas", pd.__version__, "| seaborn", sns.__version__)

pandas 3.0.5 | seaborn 0.13.2


## 1. Load the raw data

The UCI file is semicolon-separated. Loading lives in a small function so every later step starts from the same, unmodified source.

In [2]:
def load_bank_data(path: Path) -> pd.DataFrame:
    """Load the raw UCI Bank Marketing file (semicolon-separated) without modifying any values.

    Parameters
    ----------
    path : Path
        Location of ``bank-additional-full.csv``.

    Returns
    -------
    pd.DataFrame
        The raw table exactly as stored on disk.
    """
    return pd.read_csv(path, sep=";")


raw = load_bank_data(DATA_PATH)
print("Shape:", raw.shape)
raw.head()

Shape: (41188, 21)


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,duration,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,261,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,149,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,226,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,151,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,307,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


In [3]:
raw.dtypes.to_frame("dtype").T

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,duration,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
dtype,int64,str,str,str,str,str,str,str,str,str,int64,int64,int64,int64,str,float64,float64,float64,float64,float64,str


## 2. Data-quality audit

Before cleaning anything, I measure the problems: true missing values (`NaN`), the string `"unknown"` that the bank used for missing categorical answers, exact duplicate rows, and the `pdays = 999` placeholder.

In [4]:
def data_quality_report(df: pd.DataFrame) -> pd.DataFrame:
    """Summarise data-quality issues per column.

    Counts NaN values and the literal string ``"unknown"`` (the dataset's missing-value code for
    categorical answers), and reports each column's dtype and number of distinct values.

    Parameters
    ----------
    df : pd.DataFrame
        Table to audit.

    Returns
    -------
    pd.DataFrame
        One row per column with ``dtype``, ``n_unique``, ``n_nan``, ``n_unknown`` and ``pct_unknown``.
    """
    report = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "n_unique": df.nunique(),
        "n_nan": df.isna().sum(),
        "n_unknown": (df == "unknown").sum(),
    })
    report["pct_unknown"] = (100 * report["n_unknown"] / len(df)).round(2)
    return report.sort_values("n_unknown", ascending=False)


quality = data_quality_report(raw)
print("Exact duplicate rows:", raw.duplicated().sum())
print("Rows with pdays == 999:", (raw["pdays"] == 999).sum())
quality[quality["n_unknown"] > 0]

Exact duplicate rows: 12
Rows with pdays == 999: 39673


,dtype,n_unique,n_nan,n_unknown,pct_unknown
default,str,3,0,8597,20.87
education,str,8,0,1731,4.20
housing,str,3,0,990,2.40
loan,str,3,0,990,2.40
job,str,12,0,330,0.80
marital,str,4,0,80,0.19


In [5]:
# Are the 'unknown' answers for housing and loan the same clients? And does pdays == 999 mean 'never contacted'?
print("housing unknown & loan unknown on the same rows:",
      ((raw["housing"] == "unknown") & (raw["loan"] == "unknown")).sum(), "of", (raw["housing"] == "unknown").sum())
print("\nCredit in default ('default' column):")
print(raw["default"].value_counts())
print("\npdays == 999 vs previous campaign outcome:")
print(pd.crosstab(raw["pdays"] == 999, raw["poutcome"]))
print("\nprevious > 0 vs previous campaign outcome:")
print(pd.crosstab(raw["previous"] > 0, raw["poutcome"]))

housing unknown & loan unknown on the same rows: 990 of 990

Credit in default ('default' column):
default
no         32588
unknown     8597
yes            3
Name: count, dtype: int64

pdays == 999 vs previous campaign outcome:
poutcome  failure  nonexistent  success
pdays                                  
False         142            0     1373
True         4110        35563        0

previous > 0 vs previous campaign outcome:
poutcome  failure  nonexistent  success
previous                               
False           0        35563        0
True         4252            0     1373


**What the audit shows:** there are no `NaN` cells, but six categorical columns use `"unknown"`. The `default` column is the extreme case: almost all clients are `"no"` or `"unknown"` and only a handful are `"yes"`. Housing and loan are unknown for exactly the same clients. There are a few exact duplicate rows. `pdays == 999` is a placeholder rather than a real number of days. It covers every client who was never contacted before (`poutcome = "nonexistent"`), but also clients whose previous contact *failed*, for whom no day count was recorded. So "previously contacted" has to be read from `previous > 0`, which matches `poutcome != "nonexistent"` exactly, and not from `pdays`.

## 3. Cleaning functions

Each decision is implemented as a separate, documented function so it can be tested and reused.

In [6]:
def standardize_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Make column names Python-friendly and encode the target as 0/1.

    Dots in names such as ``emp.var.rate`` are replaced by underscores, and the text target
    ``y`` ('yes'/'no') becomes the integer column ``subscribed`` (1 = subscribed).

    Parameters
    ----------
    df : pd.DataFrame
        Raw table.

    Returns
    -------
    pd.DataFrame
        Copy with renamed columns and a binary ``subscribed`` target (``y`` removed).
    """
    out = df.copy()
    out.columns = [c.strip().replace(".", "_") for c in out.columns]
    out["subscribed"] = (out.pop("y") == "yes").astype(int)
    return out


def remove_duplicates(df: pd.DataFrame) -> pd.DataFrame:
    """Drop exact duplicate rows, keeping the first occurrence.

    With no client ID in the data, identical rows on all 21 columns are treated as accidental
    double entries; keeping them would slightly over-weight those contacts.

    Parameters
    ----------
    df : pd.DataFrame

    Returns
    -------
    pd.DataFrame
        De-duplicated copy with a fresh integer index.
    """
    return df.drop_duplicates(keep="first").reset_index(drop=True)


def handle_unknowns(df: pd.DataFrame, drop_cols=("job", "marital"), max_drop_pct: float = 1.0) -> pd.DataFrame:
    """Treat the ``"unknown"`` code column by column.

    * Columns in ``drop_cols`` whose share of ``"unknown"`` is at most ``max_drop_pct`` percent have
      those rows removed: the loss is negligible and the categories are not meaningful as 'unknown'.
    * All other ``"unknown"`` values are kept as an explicit category, because the missingness can
      itself be informative (for example a client not disclosing a credit default) and imputing a
      guessed value would invent information.

    Parameters
    ----------
    df : pd.DataFrame
    drop_cols : tuple of str
        Columns where rows with ``"unknown"`` may be dropped.
    max_drop_pct : float
        Safety limit: a column is only dropped-on if its ``"unknown"`` share is <= this percentage.

    Returns
    -------
    pd.DataFrame
        Copy with the selected rows removed; remaining ``"unknown"`` values preserved as a category.
    """
    out = df.copy()
    for col in drop_cols:
        pct = 100 * (out[col] == "unknown").mean()
        if pct <= max_drop_pct:
            out = out[out[col] != "unknown"]
        else:
            print(f"Kept 'unknown' in {col}: {pct:.2f}% exceeds the {max_drop_pct}% drop limit")
    return out.reset_index(drop=True)


def encode_pdays(df: pd.DataFrame, sentinel: int = 999) -> pd.DataFrame:
    """Replace the ``pdays`` placeholder with a missing value and add an explicit contact flag.

    ``pdays`` counts days since the client was last contacted in a previous campaign; ``999`` is a
    placeholder (no day count recorded). Leaving 999 in place would distort means and plots, so it
    becomes ``NaN``. Whether a client was contacted before is taken from ``previous > 0`` (the audit
    showed that some previously contacted clients also have ``pdays == 999``).

    Parameters
    ----------
    df : pd.DataFrame
    sentinel : int
        Placeholder value used for 'not previously contacted'.

    Returns
    -------
    pd.DataFrame
        Copy with ``previously_contacted`` (bool) and ``pdays`` as float with NaN for the sentinel.
    """
    out = df.copy()
    out["previously_contacted"] = out["previous"] > 0
    out["pdays"] = out["pdays"].where(out["pdays"] != sentinel).astype(float)
    return out


def cap_outliers(df: pd.DataFrame, col: str, upper_quantile: float = 0.99) -> pd.DataFrame:
    """Add a capped (winsorised) copy of a heavy-tailed column without deleting rows.

    Extreme values of ``campaign`` (number of calls in this campaign) are real but rare; they are
    kept in the original column and a ``<col>_capped`` version is clipped at the chosen quantile
    for summaries and plots.

    Parameters
    ----------
    df : pd.DataFrame
    col : str
        Numeric column to cap.
    upper_quantile : float
        Quantile used as the upper limit.

    Returns
    -------
    pd.DataFrame
        Copy with an extra ``<col>_capped`` column.
    """
    out = df.copy()
    limit = out[col].quantile(upper_quantile)
    out[f"{col}_capped"] = out[col].clip(upper=limit)
    print(f"{col}: capped at the {upper_quantile:.0%} quantile = {limit:.0f}; "
          f"{(out[col] > limit).sum()} values affected")
    return out


def set_categorical_types(df: pd.DataFrame) -> pd.DataFrame:
    """Convert text columns to pandas ``category`` dtype (smaller memory, explicit levels).

    Month and weekday get their natural calendar order so plots and tables sort correctly.
    """
    out = df.copy()
    month_order = ["mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec"]
    day_order = ["mon", "tue", "wed", "thu", "fri"]
    text_cols = [c for c in out.columns if out[c].dtype == object or pd.api.types.is_string_dtype(out[c])]
    for col in text_cols:
        if col == "month":
            out[col] = pd.Categorical(out[col], categories=month_order, ordered=True)
        elif col == "day_of_week":
            out[col] = pd.Categorical(out[col], categories=day_order, ordered=True)
        else:
            out[col] = out[col].astype("category")
    return out

In [7]:
def clean_bank_data(df: pd.DataFrame) -> pd.DataFrame:
    """Run the full cleaning pipeline and log how many rows each step keeps."""
    steps = [
        ("standardize_columns", standardize_columns),
        ("remove_duplicates", remove_duplicates),
        ("handle_unknowns", handle_unknowns),
        ("encode_pdays", encode_pdays),
        ("cap_outliers(campaign)", lambda d: cap_outliers(d, "campaign")),
        ("set_categorical_types", set_categorical_types),
    ]
    out = df
    for name, fn in steps:
        before = len(out)
        out = fn(out)
        print(f"{name:<24} rows {before:>6,} -> {len(out):>6,}")
    return out


clean = clean_bank_data(raw)
print(f"\nRows removed in total: {len(raw) - len(clean):,} ({100 * (len(raw) - len(clean)) / len(raw):.2f}%)")
clean.head()

standardize_columns      rows 41,188 -> 41,188


remove_duplicates        rows 41,188 -> 41,176
handle_unknowns          rows 41,176 -> 40,775
encode_pdays             rows 40,775 -> 40,775
campaign: capped at the 99% quantile = 14; 399 values affected
cap_outliers(campaign)   rows 40,775 -> 40,775
set_categorical_types    rows 40,775 -> 40,775

Rows removed in total: 413 (1.00%)


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,duration,campaign,pdays,previous,poutcome,emp_var_rate,cons_price_idx,cons_conf_idx,euribor3m,nr_employed,subscribed,previously_contacted,campaign_capped
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,261,1,NaN,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,False,1
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,149,1,NaN,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,False,1
2,37,services,married,high.school,no,yes,no,telephone,may,mon,226,1,NaN,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,False,1
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,151,1,NaN,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,False,1
4,56,services,married,high.school,no,no,yes,telephone,may,mon,307,1,NaN,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,False,1


In [8]:
# Sanity checks on the cleaned table
assert clean.duplicated().sum() == 0
assert not (clean["job"] == "unknown").any() and not (clean["marital"] == "unknown").any()
assert clean.loc[~clean["previously_contacted"], "pdays"].isna().all()   # never contacted -> no day count
assert (clean["previously_contacted"] == (clean["poutcome"] != "nonexistent")).all()
assert set(clean["subscribed"].unique()) == {0, 1}
print("All checks passed. Remaining 'unknown' categories kept on purpose:")
print({c: int((clean[c] == "unknown").sum()) for c in ["education", "default", "housing", "loan"]})

All checks passed. Remaining 'unknown' categories kept on purpose:
{'education': 1595, 'default': 8435, 'housing': 984, 'loan': 984}


### A note on `duration`

`duration` (length of the last call in seconds) is only known **after** the call has ended; in this file every call with a duration of 0 ended in `y = "no"` (checked below). It is fine to describe it in an exploratory analysis, but it must be excluded from any model meant to decide *whom to call*, because it leaks the outcome. Moro et al. (2014) make the same point about attributes "only known on call execution". I keep the column in the cleaned table but flag it as excluded from future modelling features.

In [9]:
LEAKAGE_COLUMNS = ["duration"]
MODEL_READY = clean.drop(columns=LEAKAGE_COLUMNS)
print("Model-ready table:", MODEL_READY.shape, "| excluded:", LEAKAGE_COLUMNS)
print("Calls with duration == 0 and their outcome:", clean.loc[clean["duration"] == 0, "subscribed"].value_counts().to_dict())

Model-ready table: (40775, 22) | excluded: ['duration']
Calls with duration == 0 and their outcome: {0: 4}
